# GFCEIP — Global Financial Crisis Early-Warning ML Pipeline

    This notebook now runs the same production training script used by the Python API, so the notebook, artifacts, docs, and API stay aligned.

    **Improvements in this version:**

    - Expanded the training window to **2000 → latest available year (up to 2025)**
    - Increased coverage to the full **World Bank universe (~206 economies)** for a much larger training set (~5k+ country-year samples after cleaning)
    - Optimizes **F1** with nested threshold tuning instead of using a fixed 0.5 cutoff
    - Uses stronger regularization and explicit overfitting checks to keep the model stable

## 0. Load and run the production trainer

In [ ]:
import importlib.util
    import json
    from pathlib import Path

    trainer_path = Path("../python-service/train_model.py").resolve()
    spec = importlib.util.spec_from_file_location("gfceip_train_model", trainer_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    run_training = module.run_training

In [ ]:
results = run_training()
    summary = {
        "selected_model": results["selected_model"],
        "cv_f1": round(results["cv"]["f1"]["mean"], 3),
        "test_f1": round(results["test"]["f1"], 3),
        "test_auc": round(results["test"]["roc_auc"], 3),
        "threshold": round(results["threshold"], 3),
        "n_samples": results["n_samples"],
        "year_range_actual": results["year_range_actual"],
    }
    print(json.dumps(summary, indent=2))

## 1. Inspect the saved metrics

In [ ]:
metrics_path = Path("../python-service/app/artifacts/metrics.json")
    metrics = json.loads(metrics_path.read_text())
    {
        "selected_model": metrics["selected_model"],
        "cv_f1": metrics["cv"]["f1"],
        "test": metrics["test"],
        "year_range_actual": metrics["year_range_actual"],
    }

## 2. Confirm generated artifacts

In [ ]:
artifacts = sorted(Path("../python-service/app/artifacts").glob("*"))
    [(p.name, p.stat().st_size) for p in artifacts]